In [1]:
print("Here begins the hybrid models notebook")

Here begins the hybrid models notebook


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_selection import SelectFromModel

from xgboost import XGBClassifier

# Load data, display first few rows
pd.set_option("display.max_columns", None)
df_ml = pd.read_csv("../../data/processed/mental_health_tech_ml.csv")

df_ml.head()

,Age,self_employed,family_history,treatment,remote_work,tech_company,obs_consequence,Age__missing,Gender__missing,self_employed__missing,work_interfere__missing,Gender_female,Gender_male,work_interfere_Never,work_interfere_Often,work_interfere_Rarely,work_interfere_Sometimes,no_employees_1-5,no_employees_100-500,no_employees_26-100,no_employees_500-1000,no_employees_6-25,no_employees_More than 1000,benefits_Don't know,benefits_No,benefits_Yes,care_options_No,care_options_Not sure,care_options_Yes,wellness_program_Don't know,wellness_program_No,wellness_program_Yes,seek_help_Don't know,seek_help_No,seek_help_Yes,anonymity_Don't know,anonymity_No,anonymity_Yes,leave_Don't know,leave_Somewhat difficult,leave_Somewhat easy,leave_Very difficult,leave_Very easy,mental_health_consequence_Maybe,mental_health_consequence_No,mental_health_consequence_Yes,phys_health_consequence_Maybe,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_No,coworkers_Some of them,coworkers_Yes,supervisor_No,supervisor_Some of them,supervisor_Yes,mental_health_interview_Maybe,mental_health_interview_No,mental_health_interview_Yes,phys_health_interview_Maybe,phys_health_interview_No,phys_health_interview_Yes,mental_vs_physical_Don't know,mental_vs_physical_No,mental_vs_physical_Yes
0,37.0,0.0,0,1,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,1,0,1,0,0,0,0,1
1,44.0,0.0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,1,0,0
2,32.0,0.0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,1,0,1,0
3,31.0,0.0,1,1,0,1,1,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,1,0,0,1,0,0,1,0,0,0,1,0
4,31.0,0.0,0,0,1,1,0,0,0,1,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,1,1,0,0


In [2]:
# Target variable: work interference + treatment
# Exclude rows where the original target-defining variables were missing
df = df_ml.copy()

# Keep only rows with observed work_interfere and treatment
if "work_interfere__missing" in df.columns:
    df = df[df["work_interfere__missing"] == 0].copy()

if "treatment__missing" in df.columns:
    df = df[df["treatment__missing"] == 0].copy()

# Create binary work interference variable
# 0 = Never
# 1 = Rarely / Sometimes / Often
df["work_interfere_binary"] = (
    df[[
        "work_interfere_Rarely",
        "work_interfere_Sometimes",
        "work_interfere_Often"
    ]].sum(axis=1) > 0
).astype(int)

# Create combined target variable:
# 0 = no interference
# 1 = interference but no treatment
# 2 = interference + treatment
df["combined_target"] = np.select(
    [
        df["work_interfere_Never"] == 1,
        (df["work_interfere_binary"] == 1) & (df["treatment"] == 0),
        (df["work_interfere_binary"] == 1) & (df["treatment"] == 1),
    ],
    [0, 1, 2],
    default=np.nan
)

# Drop any rows that still do not have a valid target
df = df.dropna(subset=["combined_target"]).copy()
df["combined_target"] = df["combined_target"].astype(int)

# Define predictors and target
# Remove:
# - target variable
# - target-defining variables (to avoid leakage)
# - missingness indicators for target-defining variables
X = df.drop(columns=[
    "combined_target",
    "treatment",
    "work_interfere_Never",
    "work_interfere_Rarely",
    "work_interfere_Sometimes",
    "work_interfere_Often",
    "work_interfere__missing",
    "work_interfere_binary",
    "treatment__missing"
], errors="ignore")

# Reference categories to drop (one per categorical group)
# Choice: drop the "baseline" or most common category per group
REFERENCE_COLS = [
    "Gender_male",                          # reference: male
    "no_employees_1-5",                     # reference: smallest company
    "benefits_No",                          # reference: no benefits
    "care_options_No",                      # reference: no care options
    "wellness_program_No",                  # reference: no wellness program
    "seek_help_No",                         # reference: no seek help info
    "anonymity_No",                         # reference: no anonymity
    "leave_Very difficult",                 # reference: hardest to take leave
    "mental_health_consequence_No",         # reference: no consequence
    "phys_health_consequence_No",
    "coworkers_No",
    "supervisor_No",
    "mental_health_interview_No",
    "phys_health_interview_No",
    "mental_vs_physical_No",
]

X = X.drop(columns=REFERENCE_COLS, errors="ignore")

y = df["combined_target"]

# Check final class distribution
print(y.value_counts())
print(y.value_counts(normalize=True))
print(X.shape, y.shape)

combined_target
2    603
0    213
1    179
Name: count, dtype: int64
combined_target
2    0.606030
0    0.214070
1    0.179899
Name: proportion, dtype: float64
(995, 43) (995,)


In [3]:
# Split the dataset into training and test sets
# - test_size=0.2 means 80% training, 20% testing
# - random_state ensures reproducibility
# - stratify=y preserves the class distribution in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display shapes of training and test sets
# Confirms correct split and number of features
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Display normalized class distribution in the training set
# Verify that stratification preserved class proportions
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

Train shape: (796, 43)
Test shape: (199, 43)

Train class distribution:
combined_target
2    0.606784
0    0.213568
1    0.179648
Name: proportion, dtype: float64


In [4]:
# Initialize Random Forest model to be used for feature selection
# - n_estimators: number of trees in the forest
# - max_depth, min_samples_*: control model complexity and reduce overfitting
# - max_features="sqrt": standard choice for classification trees
# - class_weight="balanced": adjusts for class imbalance
# - n_jobs=-1: uses all available CPU cores
rf_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# Fit the Random Forest model on the training data
# This model will learn feature importances based on nonlinear relationships
rf_selector.fit(X_train, y_train)

# Initialize feature selector using the trained Random Forest
# - threshold="median": keep features with importance above the median importance
# This results in selecting approximately 50% of the most important features
selector = SelectFromModel(
    rf_selector,
    threshold="median"
)

# Fit the selector to identify important features
selector.fit(X_train, y_train)

# Transform training and test sets to keep only selected features
X_train_sel = selector.transform(X_train)
X_test_sel = selector.transform(X_test)

# Retrieve names of selected features
selected_features = X_train.columns[selector.get_support()]

# Display number and list of selected features
print("Number of selected features:", len(selected_features))
print("\nSelected features:")
print(selected_features.tolist())

Number of selected features: 22

Selected features:
['Age', 'family_history', 'remote_work', 'Gender_female', 'no_employees_26-100', 'no_employees_6-25', "benefits_Don't know", 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', "seek_help_Don't know", "anonymity_Don't know", 'anonymity_Yes', "leave_Don't know", 'leave_Somewhat easy', 'mental_health_consequence_Maybe', 'mental_health_consequence_Yes', 'coworkers_Some of them', 'supervisor_Some of them', 'supervisor_Yes', 'phys_health_interview_Maybe', "mental_vs_physical_Don't know"]


In [5]:
# Initialize Logistic Regression model for the hybrid approach
# - max_iter increased to ensure convergence
# - class_weight="balanced": compensates for class imbalance
# - this model serves as the interpretable component of the hybrid approach
logreg_hybrid = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train the Logistic Regression model on the selected features
# These features were previously chosen by a nonlinear model (Random Forest)
logreg_hybrid.fit(X_train_sel, y_train)

# Generate predictions on the test set
y_pred_hybrid1 = logreg_hybrid.predict(X_test_sel)

# Evaluate model performance
# - Accuracy: overall correctness
# - Macro F1: treats all classes equally
# - Weighted F1: accounts for class frequency
print("Hybrid 1: RF Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid1))
print("Macro F1:", f1_score(y_test, y_pred_hybrid1, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid1, average="weighted"))

# Detailed classification report:
# shows precision, recall, and F1-score for each class separately
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid1))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# useful for identifying which classes are confused with each other
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid1))

Hybrid 1: RF Feature Selection -> Logistic Regression
Accuracy: 0.542713567839196
Macro F1: 0.5002751100440176
Weighted F1: 0.5660037884500534

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.56      0.49        43
           1       0.28      0.47      0.35        36
           2       0.80      0.56      0.66       120

    accuracy                           0.54       199
   macro avg       0.51      0.53      0.50       199
weighted avg       0.63      0.54      0.57       199


Confusion Matrix:

[[24 10  9]
 [11 17  8]
 [20 33 67]]


In [6]:
# Create a DataFrame of model coefficients from the multinomial Logistic Regression
# - rows correspond to classes (class_0, class_1, class_2)
# - columns correspond to the selected features
# - coefficients represent the effect of each feature on the log-odds of belonging to a class
coef_df = pd.DataFrame(
    logreg_hybrid.coef_,
    columns=selected_features,
    index=[f"class_{c}" for c in logreg_hybrid.classes_]
)

# Transpose the DataFrame so that features are rows and classes are columns
# Then sort features by the absolute value of their coefficient for class_2
# - class_2 corresponds to "interference + treatment"
# - sorting by absolute value highlights the most influential predictors (positive or negative)
# Finally, display the top 15 most important features for class_2
coef_df.T.sort_values(by="class_2", key=np.abs, ascending=False).head(15)

,class_0,class_1,class_2
family_history,-0.868073,-0.005441,0.873514
care_options_Yes,-0.335837,-0.094630,0.430467
Gender_female,-0.140425,-0.248412,0.388837
leave_Somewhat easy,0.056957,0.320091,-0.377048
leave_Don't know,0.370792,0.006245,-0.377037
mental_health_consequence_Yes,-0.323263,0.067355,0.255908
care_options_Not sure,-0.008375,0.229338,-0.220963
no_employees_26-100,0.092388,-0.303017,0.210629
anonymity_Yes,0.010566,-0.197946,0.187381
no_employees_6-25,-0.254188,0.072052,0.182136


In [7]:
# Initialize XGBoost classifier to be used for feature selection
# - n_estimators: number of trees
# - max_depth: controls tree complexity
# - learning_rate: step size for boosting
# - subsample, colsample_bytree: introduce randomness and reduce overfitting
# - objective="multi:softprob": multiclass classification with probability outputs
# - num_class=3: number of target classes
# - eval_metric="mlogloss": suitable loss for multiclass classification
# - tree_method="hist": efficient histogram-based algorithm
xgb_selector = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

# Fit XGBoost model on training data
# The model learns nonlinear relationships and assigns importance scores to features
xgb_selector.fit(X_train, y_train)

# Initialize feature selector using the trained XGBoost model
# - threshold="median": retains features with importance above the median
selector_xgb = SelectFromModel(
    xgb_selector,
    threshold="median"
)

# Fit selector to determine which features to keep
selector_xgb.fit(X_train, y_train)

# Transform training and test data to include only selected features
# This reduces dimensionality and keeps the most informative predictors
X_train_sel_xgb = selector_xgb.transform(X_train)
X_test_sel_xgb = selector_xgb.transform(X_test)

# Retrieve names of selected features
selected_features_xgb = X_train.columns[selector_xgb.get_support()]

# Display number and list of selected features
# Useful for comparison with Random Forest-based feature selection
print("Number of selected features:", len(selected_features_xgb))
print(selected_features_xgb.tolist())

Number of selected features: 22
['Age', 'family_history', 'Gender_female', 'no_employees_500-1000', 'no_employees_More than 1000', "benefits_Don't know", 'benefits_Yes', 'care_options_Not sure', 'care_options_Yes', 'wellness_program_Yes', "seek_help_Don't know", 'seek_help_Yes', "anonymity_Don't know", 'anonymity_Yes', "leave_Don't know", 'leave_Somewhat difficult', 'leave_Somewhat easy', 'mental_health_consequence_Maybe', 'mental_health_consequence_Yes', 'coworkers_Yes', 'mental_health_interview_Maybe', 'phys_health_interview_Yes']


In [8]:
# Initialize Logistic Regression model for the second hybrid approach
# - Uses features selected by XGBoost (nonlinear model)
# - max_iter increased to ensure convergence
# - class_weight="balanced": handles class imbalance
logreg_hybrid_xgb = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

# Train Logistic Regression on XGBoost-selected features
# This combines nonlinear feature selection with a linear, interpretable model
logreg_hybrid_xgb.fit(X_train_sel_xgb, y_train)

# Generate predictions on the test set
y_pred_hybrid2 = logreg_hybrid_xgb.predict(X_test_sel_xgb)

# Evaluate model performance
# - Accuracy: overall correctness
# - Macro F1: treats all classes equally
# - Weighted F1: accounts for class distribution
print("Hybrid 2: XGB Feature Selection -> Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_hybrid2))
print("Macro F1:", f1_score(y_test, y_pred_hybrid2, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_hybrid2, average="weighted"))

# Detailed classification report:
# shows precision, recall, and F1-score for each class
# useful for identifying which classes are harder to predict
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_hybrid2))

# Confusion matrix:
# provides insight into misclassification patterns between classes
# helps analyze where the model makes systematic errors
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_hybrid2))

Hybrid 2: XGB Feature Selection -> Logistic Regression
Accuracy: 0.5577889447236181
Macro F1: 0.5119966230476994
Weighted F1: 0.5796355776349031

Classification Report:

              precision    recall  f1-score   support

           0       0.47      0.63      0.54        43
           1       0.27      0.42      0.33        36
           2       0.80      0.57      0.67       120

    accuracy                           0.56       199
   macro avg       0.51      0.54      0.51       199
weighted avg       0.63      0.56      0.58       199


Confusion Matrix:

[[27  8  8]
 [12 15  9]
 [18 33 69]]


In [12]:
coef_df_xgb = pd.DataFrame(
    logreg_hybrid_xgb.coef_,
    columns=selected_features_xgb,
    index=[f"class_{c}" for c in logreg_hybrid_xgb.classes_]
)

coef_df_xgb.T.sort_values(
    by="class_2", key=np.abs, ascending=False
).head(15)

,class_0,class_1,class_2
family_history,-0.875407,-0.010548,0.885955
no_employees_500-1000,0.557523,-0.042828,-0.514695
phys_health_interview_Yes,-0.259205,-0.207713,0.466917
care_options_Yes,-0.298683,-0.164412,0.463095
Gender_female,-0.207370,-0.233364,0.440735
coworkers_Yes,-0.097907,-0.318895,0.416802
leave_Don't know,0.412453,-0.071563,-0.340889
leave_Somewhat easy,0.099073,0.229451,-0.328524
mental_health_consequence_Yes,-0.443123,0.130235,0.312888
benefits_Yes,0.157177,-0.463142,0.305965


In [9]:
# Define base models for stacking
# These represent different modeling paradigms:
# - Logistic Regression - linear, interpretable model
# - Random Forest - nonlinear, ensemble tree model
# - XGBoost - gradient boosting model with high predictive power
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

# Initialize stacking classifier
# - base estimators generate predictions
# - final_estimator learns how to combine these predictions
# - stack_method="predict_proba": uses predicted probabilities as input to meta-model
# - cv=5: cross-validation used to generate out-of-fold predictions for training meta-model
stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=2000, random_state=42),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

# Train stacking model on training data
# The meta-model learns how to optimally combine base model outputs
stack_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_stack = stack_model.predict(X_test)

# Evaluate performance of stacking hybrid model
# - Accuracy: overall correctness
# - Macro F1: equal importance to all classes
# - Weighted F1: accounts for class distribution
print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))

# Detailed classification report:
# shows per-class precision, recall, and F1-score
# useful for understanding performance differences across classes
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# helps identify systematic misclassification patterns
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.628140703517588
Macro F1: 0.3926206813530757
Weighted F1: 0.551526139182732

Classification Report:

              precision    recall  f1-score   support

           0       0.46      0.37      0.41        43
           1       0.00      0.00      0.00        36
           2       0.66      0.91      0.77       120

    accuracy                           0.63       199
   macro avg       0.37      0.43      0.39       199
weighted avg       0.50      0.63      0.55       199


Confusion Matrix:

[[ 16   0  27]
 [  8   0  28]
 [ 11   0 109]]


D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
D:\Diana\Anul III\BachelorArbeit\WellBeingModelling\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p

In [10]:
# Define base models for stacking, same approach as before
base_estimators = [
    ("lr", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",   # handles class imbalance at base level
        random_state=42
    )),
    ("rf", RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        class_weight="balanced",   # adjusts for imbalance
        random_state=42,
        n_jobs=-1
    )),
    ("xgb", XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
]

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(
        max_iter=2000,
        class_weight="balanced"   # difference - add balanced for avoiding bias toward majority classes
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

# Train stacking model
# The meta-model learns how to combine base model predictions
stack_model.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_stack = stack_model.predict(X_test)

# Evaluate performance
# - Accuracy: overall correctness
# - Macro F1: equal importance to all classes (key metric for imbalanced data)
# - Weighted F1: weighted by class frequency
print("Hybrid 3: Stacking")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Macro F1:", f1_score(y_test, y_pred_stack, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_stack, average="weighted"))

# Classification report:
# shows precision, recall, and F1-score for each class
# useful for analyzing class-specific performance
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stack))

# Confusion matrix:
# shows how predictions are distributed across true vs predicted classes
# helps identify misclassification patterns and class overlap
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_stack))

Hybrid 3: Stacking
Accuracy: 0.5477386934673367
Macro F1: 0.473935296174619
Weighted F1: 0.5739519313004283

Classification Report:

              precision    recall  f1-score   support

           0       0.34      0.56      0.42        43
           1       0.26      0.31      0.28        36
           2       0.85      0.62      0.71       120

    accuracy                           0.55       199
   macro avg       0.49      0.49      0.47       199
weighted avg       0.63      0.55      0.57       199


Confusion Matrix:

[[24 12  7]
 [19 11  6]
 [27 19 74]]


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "f1_macro":    make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
}

hybrid_models = {
    "RF Selection → LogReg":    logreg_hybrid,
    "XGB Selection → LogReg":   logreg_hybrid_xgb,
    "Stacking (LR + RF + XGB)": stack_model,
}

cv_results_hybrid = {}
for name, model in hybrid_models.items():
    # For feature-selected models, CV must use the already-transformed X
    # Stacking uses full X
    if name == "RF Selection → LogReg":
        X_cv = pd.DataFrame(X_train_sel, columns=selected_features) if hasattr(X_train_sel, '__len__') else X_train_sel
        # Simpler: refit on full selected feature matrix
        X_cv_full = selector.transform(X)
        scores = cross_validate(model, X_cv_full, y, cv=cv, scoring=scoring, n_jobs=-1)
        y_prob_cv = cross_val_predict(model, X_cv_full, y, cv=cv, method="predict_proba", n_jobs=-1)
    elif name == "XGB Selection → LogReg":
        X_cv_full = selector_xgb.transform(X)
        scores = cross_validate(model, X_cv_full, y, cv=cv, scoring=scoring, n_jobs=-1)
        y_prob_cv = cross_val_predict(model, X_cv_full, y, cv=cv, method="predict_proba", n_jobs=-1)
    else:
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        y_prob_cv = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=-1)

    auc = roc_auc_score(y, y_prob_cv, multi_class="ovr", average="macro")

    cv_results_hybrid[name] = {
        "F1 Macro (mean)":    round(scores["test_f1_macro"].mean(), 3),
        "F1 Macro (std)":     round(scores["test_f1_macro"].std(),  3),
        "F1 Weighted (mean)": round(scores["test_f1_weighted"].mean(), 3),
        "F1 Weighted (std)":  round(scores["test_f1_weighted"].std(),  3),
        "AUC OvR":            round(auc, 3),
    }

cv_hybrid_df = pd.DataFrame(cv_results_hybrid).T
print(cv_hybrid_df)

                          F1 Macro (mean)  F1 Macro (std)  F1 Weighted (mean)  \
RF Selection → LogReg               0.465           0.025               0.553   
XGB Selection → LogReg              0.488           0.021               0.577   
Stacking (LR + RF + XGB)            0.460           0.017               0.560   

                          F1 Weighted (std)  AUC OvR  
RF Selection → LogReg                 0.022    0.701  
XGB Selection → LogReg                0.022    0.701  
Stacking (LR + RF + XGB)              0.013    0.700  


In [11]:
# Create a summary DataFrame to compare hybrid models
# Each row corresponds to one hybrid approach and its evaluation metrics
results = pd.DataFrame([
    {
        "Model": "RF selection -> Logistic Regression",
        # Overall accuracy of the hybrid model
        "Accuracy": accuracy_score(y_test, y_pred_hybrid1),
        # Macro F1: gives equal importance to all classes
        "Macro F1": f1_score(y_test, y_pred_hybrid1, average="macro"),
        # Weighted F1: accounts for class distribution
        "Weighted F1": f1_score(y_test, y_pred_hybrid1, average="weighted"),
    },
    {
        "Model": "XGB selection -> Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred_hybrid2),
        "Macro F1": f1_score(y_test, y_pred_hybrid2, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_hybrid2, average="weighted"),
    },
    {
        "Model": "Stacking (LR + RF + XGB)",
        "Accuracy": accuracy_score(y_test, y_pred_stack),
        "Macro F1": f1_score(y_test, y_pred_stack, average="macro"),
        "Weighted F1": f1_score(y_test, y_pred_stack, average="weighted"),
    }
])

# Sort models by Macro F1 score (descending)
# This highlights the best model in terms of balanced performance across classes
results.sort_values("Macro F1", ascending=False)

,Model,Accuracy,Macro F1,Weighted F1
1,XGB selection -> Logistic Regression,0.557789,0.511997,0.579636
0,RF selection -> Logistic Regression,0.542714,0.500275,0.566004
2,Stacking (LR + RF + XGB),0.547739,0.473935,0.573952


In [14]:
results_hybrid = pd.DataFrame([
    {
        "Model":              "RF Selection → LogReg",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_hybrid1, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_hybrid1, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["RF Selection → LogReg", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["RF Selection → LogReg", "F1 Macro (std)"],
        "AUC OvR":            cv_hybrid_df.loc["RF Selection → LogReg", "AUC OvR"],
    },
    {
        "Model":              "XGB Selection → LogReg",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_hybrid2, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_hybrid2, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["XGB Selection → LogReg", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["XGB Selection → LogReg", "F1 Macro (std)"],
        "AUC OvR":            cv_hybrid_df.loc["XGB Selection → LogReg", "AUC OvR"],
    },
    {
        "Model":              "Stacking (LR + RF + XGB)",
        "Macro F1 (test)":    round(f1_score(y_test, y_pred_stack, average="macro"), 3),
        "Weighted F1 (test)": round(f1_score(y_test, y_pred_stack, average="weighted"), 3),
        "CV Macro F1":        cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "F1 Macro (mean)"],
        "CV Macro F1 std":    cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "F1 Macro (std)"],
        "AUC OvR":            cv_hybrid_df.loc["Stacking (LR + RF + XGB)", "AUC OvR"],
    },
]).set_index("Model")

print(results_hybrid)

                          Macro F1 (test)  Weighted F1 (test)  CV Macro F1  \
Model                                                                        
RF Selection → LogReg               0.500               0.566        0.465   
XGB Selection → LogReg              0.512               0.580        0.488   
Stacking (LR + RF + XGB)            0.474               0.574        0.460   

                          CV Macro F1 std  AUC OvR  
Model                                               
RF Selection → LogReg               0.025    0.701  
XGB Selection → LogReg              0.021    0.701  
Stacking (LR + RF + XGB)            0.017    0.700  


In [ ]:
## Hybrid Models — Results Summary (Mental Health in Tech)

### Approach
# Three hybrid pipelines were evaluated:
# 1. RF feature selection (threshold=median, 22 features) → Logistic Regression
# 2. XGB feature selection (threshold=median, 22 features) → Logistic Regression
# 3. Stacking ensemble (LR + RF + XGB base learners, LR meta-learner, cv=5)

### Key Findings
# - XGB Selection → LogReg is the best hybrid model (CV Macro F1: 0.488,
#   AUC: 0.701), marginally outperforming the RF selection pipeline (0.465).
# - All three hybrid models achieve nearly identical AUC (~0.700), suggesting
#   the ensemble combination strategy matters less than feature quality for
#   this dataset.
# - Stacking has the lowest variance (CV std: 0.017) but does not outperform
#   the simpler selection pipelines on macro F1, suggesting the added complexity
#   is not justified here.
# - Hybrid models improve AUC over baselines (0.696 RF → 0.701 hybrids) but
#   do not substantially improve macro F1, indicating that feature selection
#   adds discriminative value primarily for the majority class.
# - The gap between test Macro F1 and CV Macro F1 remains consistent (~0.02–0.03),
#   similar to baseline models, suggesting stable evaluation across splits.

### Comparison with Baseline
# Hybrid models do not substantially outperform baseline models on this dataset.
# The best baseline (RF, CV Macro F1: 0.470) and best hybrid (XGB→LogReg,
# CV Macro F1: 0.488) are comparable. This is a meaningful finding — it suggests
# that for this dataset, feature selection and ensembling add limited value beyond
# a well-configured single model.